# Set Packing Problem

It's similar to tutorial007_exact_cover_en.ipynb

Set packing is a classical NP-complete problem in computational complexity theory and combinatorics, and was one of Karp's 21 NP-complete problems.

Suppose one has a finite set S and a list of subsets of S. Then, the set packing problem asks if some k subsets in the list are pairwise disjoint (in other words, no two of them share an element).

https://en.wikipedia.org/wiki/Set_packing

## Installation

Please install blueqat first

```bash
pip install git+https://github.com/blueqat/blueqatSDK
```

Import libraries and prepare instance.

In [1]:
import numpy as np
from blueqat.utils import Vqe, QaoaAnsatz
from blueqat.utils import qubo_bit as q

## create a QUBO

This time we have the cost function 

$ E = E_{A} + E_{B} $

And each of $E_{A}, E_{B}$ are defined as 

$ E _ { A } = A \sum _ { i , j : V _ { i } \cap V _ { j } \neq \emptyset } x _ { i } x _ { j }$

$E _ { B } = - B \sum _ { i } x _ { i }$

For $A, B$  we need $A > B$

In [2]:
A = 1.0
B = 0.9

def get_qubo(V):
    Q = 0

    for i in range(len(V)):
        for j in range(i, len(V)):
            if i == j:
                Q += -B*q(i)*q(i)
            elif len(V[i]) + len(V[j]) != len( set(V[i]) | set(V[j]) ):
                Q +=  A*q(i)*q(j)

    return Q

And function for showing result.

In [3]:
def show_answer(list_x, energies = None, show_graph = False):
    print("Result x:", list_x)
    text = ""
    for i in range(len(list_x)):
        if(list_x[i]):
            text += str(V[i])
    print("Picked {} group(s): {}".format(sum(list_x), text))
    if energies is not None:
        print("Energy:", a.E[-1])
    if show_graph:
        plt.plot(a.E)
        plt.show()

Let's start it.

In [4]:
V = [ [1,2], [3,4,5,6], [7,8,9,10], [1,3,5], [10], [7,9], [2,4,6,8], [1,2,3,4,5,6,8,10] ]

h = get_qubo(V)
# Note: in the new SDK, `step` is the number of variational QAOA layers
# actually optimized by gradient descent (unlike the old Trotter-schedule
# semantics), so a small step count is used here to keep runtime reasonable.
step = 5
result = Vqe(QaoaAnsatz(h, step)).run(max_iter=100)
print(result.most_common(12))

/Users/yuichirominato/blueqatSDK/.claude/worktrees/determined-mahavira-bf713e/blueqat/utils.py:399: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/Context.cpp:823.)
  total_matrix = torch.sparse_coo_tensor(torch.empty((2, 0), dtype=torch.int64, device=device), torch.empty(0, dtype=torch.complex128, device=device), (dim, dim))


(((1, 1, 0, 0, 1, 1, 0, 0), 0.20270217794900786), ((0, 0, 0, 1, 1, 1, 1, 0), 0.10077524415864446), ((0, 0, 0, 1, 1, 1, 0, 0), 0.0807417631189329), ((0, 1, 0, 1, 1, 1, 0, 0), 0.03994927015141452), ((1, 0, 0, 1, 1, 1, 0, 0), 0.03994927015141449), ((1, 1, 1, 0, 0, 1, 0, 0), 0.0362253534349444), ((1, 1, 0, 0, 0, 1, 0, 0), 0.0324630124368081), ((0, 1, 0, 0, 1, 1, 0, 0), 0.02943919130302158), ((1, 0, 0, 0, 1, 1, 0, 0), 0.02943919130302156), ((1, 0, 0, 0, 1, 1, 1, 0), 0.02829871061921266), ((0, 1, 0, 0, 1, 1, 1, 0), 0.02829871061921266), ((0, 0, 0, 1, 0, 1, 1, 0), 0.027853978370676952))


The correct answer is {1,3,5},{10},{7,9},{2,4,6,8}. Sometimes different answer selected.

This Set Packing problem has similarity to Maximal Independent Set (MIS) Problem. These 2 problems are the same.